Phase 1 — turning the raw Yelp Open Dataset into a 3-city restaurant subset with temporal train/val/test split. Holds out two evaluation subsets (cold-start users, multi-city users) before the temporal split, then extracts a cold-start *item* subset from the test partition (restaurants the model never saw during training).

In [1]:
import json
import logging
import time
from pathlib import Path

import pandas as pd

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(message)s",
                    datefmt="%H:%M:%S",
                    force=True)
log = logging.getLogger(__name__)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "yelp_dataset_link"
OUT_DIR = PROJECT_ROOT / "data" / "cleaned"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CITIES = ["Philadelphia", "Tucson", "Tampa"]
print(PROJECT_ROOT)
print(CITIES)

/Users/yanghaobo/projects/Yelp-Recommendation-System
['Philadelphia', 'Tucson', 'Tampa']


The review file is 5.4 GB so I stream it line by line instead of loading with pandas.

In [2]:
def stream_json_lines(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

City filter on the business file. Picked Philadelphia / Tucson / Tampa because they're the three cities with the most data in the Yelp Open Dataset — LA, Chicago, NYC and Seattle all have under 10 businesses each so they're not usable.

In [3]:
def filter_businesses_by_city(business_path, cities):
    cities_set = set(cities)
    rows, seen = [], 0
    for biz in stream_json_lines(business_path):
        seen += 1
        if seen % 50_000 == 0:
            log.info("scanned %d, kept %d", seen, len(rows))
        if (biz.get("city") or "").strip() in cities_set:
            rows.append(biz)
    log.info("done: kept %d of %d", len(rows), seen)
    return pd.DataFrame(rows)

t0 = time.time()
businesses = filter_businesses_by_city(RAW_DIR / "yelp_academic_dataset_business.json", CITIES)
businesses.to_parquet(OUT_DIR / "businesses_target.parquet", index=False)
print(f"businesses: {len(businesses)} rows in {time.time()-t0:.1f}s")
businesses.head(3)

02:00:44 | scanned 50000, kept 10931


02:00:44 | scanned 100000, kept 21867


02:00:44 | scanned 150000, kept 32797


02:00:44 | done: kept 32873 of 150346


businesses: 32873 rows in 0.7s


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
1,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
2,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,1,"{'RestaurantsReservations': 'True', 'Restauran...","Sushi Bars, Restaurants, Japanese","{'Tuesday': '13:30-22:0', 'Wednesday': '13:30-..."


Now stream the review file and keep reviews on those businesses.

In [4]:
def filter_reviews_by_business_ids(review_path, business_ids):
    rows, seen = [], 0
    for r in stream_json_lines(review_path):
        seen += 1
        if seen % 500_000 == 0:
            log.info("scanned %d, kept %d", seen, len(rows))
        if r.get("business_id") in business_ids:
            rows.append(r)
    log.info("done: kept %d of %d", len(rows), seen)
    return pd.DataFrame(rows)

t0 = time.time()
biz_ids = set(businesses["business_id"])
reviews = filter_reviews_by_business_ids(RAW_DIR / "yelp_academic_dataset_review.json", biz_ids)
reviews.to_parquet(OUT_DIR / "reviews_target.parquet", index=False)
print(f"reviews: {len(reviews)} rows in {time.time()-t0:.1f}s")

02:00:45 | scanned 500000, kept 133882


02:00:46 | scanned 1000000, kept 263791


02:00:47 | scanned 1500000, kept 393813


02:00:48 | scanned 2000000, kept 529685


02:00:49 | scanned 2500000, kept 669312


...

02:00:55 | scanned 5500000, kept 1439801


02:00:56 | scanned 6000000, kept 1567681


02:00:57 | scanned 6500000, kept 1698540


02:00:58 | done: kept 1827442 of 6990280


reviews: 1827442 rows in 19.1s


Restaurant subset. Yelp `categories` is comma-joined (e.g. `"Restaurants, Italian, Pizza"`), so I substring-match for `Restaurants` or `Food`. Also drop closed places — they shouldn't be in a recommendation pool.

In [5]:
cat = businesses["categories"].fillna("")
is_food = cat.str.contains("Restaurants", case=False) | cat.str.contains("Food", case=False)
is_open = businesses["is_open"] == 1
restaurants = businesses.loc[is_food & is_open].reset_index(drop=True)
restaurants.to_parquet(OUT_DIR / "restaurants_open.parquet", index=False)
print(f"restaurants: {len(restaurants)}")
print(restaurants["city"].value_counts())

restaurants: 9022
city
Philadelphia     4372
Tampa            2502
Tucson           2147
Philadelphia        1
Name: count, dtype: int64


In [6]:
rest_ids = set(restaurants["business_id"])
reviews_rest = reviews[reviews["business_id"].isin(rest_ids)].reset_index(drop=True)
reviews_rest.to_parquet(OUT_DIR / "reviews_restaurant.parquet", index=False)
print(f"reviews on restaurants: {len(reviews_rest)}")

reviews on restaurants: 1032056


User filter — keep only users who reviewed at least one of these restaurants.

In [7]:
def filter_users_by_ids(user_path, user_ids):
    rows, seen = [], 0
    for u in stream_json_lines(user_path):
        seen += 1
        if seen % 200_000 == 0:
            log.info("scanned %d, kept %d", seen, len(rows))
        if u.get("user_id") in user_ids:
            rows.append(u)
    log.info("done: kept %d of %d", len(rows), seen)
    return pd.DataFrame(rows)

t0 = time.time()
user_ids_seen = set(reviews_rest["user_id"].unique())
users = filter_users_by_ids(RAW_DIR / "yelp_academic_dataset_user.json", user_ids_seen)
users.to_parquet(OUT_DIR / "users_target.parquet", index=False)
print(f"users: {len(users)} in {time.time()-t0:.1f}s")

02:01:06 | scanned 200000, kept 63702


02:01:07 | scanned 400000, kept 111701


02:01:07 | scanned 600000, kept 156315


02:01:08 | scanned 800000, kept 195685


02:01:09 | scanned 1000000, kept 231026


...

02:01:10 | scanned 1400000, kept 290177


02:01:11 | scanned 1600000, kept 315086


02:01:12 | scanned 1800000, kept 338665


02:01:12 | done: kept 359007 of 1987897


users: 359007 in 9.1s


Cold-start subset — users with `review_count < 5`. These are held out before the temporal split so their reviews never enter training. At eval time they're a proxy for what a brand-new Taste hunter user sees with the `<NEW_USER>` OOV embedding.

In [8]:
cold_user_ids = set(users.loc[users["review_count"] < 5, "user_id"])
coldstart_test = reviews_rest[reviews_rest["user_id"].isin(cold_user_ids)].copy()
coldstart_test.to_parquet(OUT_DIR / "coldstart_test_reviews.parquet", index=False)
print(f"cold-start: {len(coldstart_test)} reviews from {len(cold_user_ids)} users")

cold-start: 140948 reviews from 119156 users


Cross-city subset — users who reviewed restaurants in two or more of our target cities. They're the closest thing we have to "travelers" in the training data, so we use them to measure cross-city generalization for the F2 trip mode.

In [9]:
biz_city = restaurants.set_index("business_id")["city"].to_dict()
rev_with_city = reviews_rest.copy()
rev_with_city["city"] = rev_with_city["business_id"].map(biz_city)
user_cities = rev_with_city.groupby("user_id")["city"].nunique()
multi_city_users = set(user_cities[user_cities >= 2].index)
crosscity_test = reviews_rest[reviews_rest["user_id"].isin(multi_city_users)].copy()
crosscity_test.to_parquet(OUT_DIR / "crosscity_test_reviews.parquet", index=False)
print(f"cross-city: {len(crosscity_test)} reviews from {len(multi_city_users)} users")

cross-city: 51720 reviews from 5250 users


Temporal split. The two held-out subsets above are excluded first, then the remainder is split by `review.date` quantile: ≤P70 train, P70–P80 val, >P80 test. Test stays locked until Phase 7. A temporal split (instead of random) keeps the past→future direction that real deployment has.

In [10]:
held_out = set(coldstart_test["user_id"]).union(set(crosscity_test["user_id"]))
main = reviews_rest[~reviews_rest["user_id"].isin(held_out)].reset_index(drop=True)
main["date"] = pd.to_datetime(main["date"])
main = main.sort_values("date").reset_index(drop=True)

cut_train = main["date"].quantile(0.70)
cut_val = main["date"].quantile(0.80)
print(f"P70 cut: {cut_train}")
print(f"P80 cut: {cut_val}")

train = main[main["date"] <= cut_train]
val = main[(main["date"] > cut_train) & (main["date"] <= cut_val)]
test = main[main["date"] > cut_val]

train.to_parquet(OUT_DIR / "train_reviews.parquet", index=False)
val.to_parquet(OUT_DIR / "val_reviews.parquet", index=False)
test.to_parquet(OUT_DIR / "test_reviews.parquet", index=False)
print(f"train={len(train)} val={len(val)} test={len(test)}")

P70 cut: 2018-12-09 21:10:43.400000
P80 cut: 2019-09-17 15:07:09.800000


train=587676 val=83953 test=167908


Cold-start ITEM subset for Phase 7 evaluation. After the temporal split is locked, find the restaurants whose `business_id` appears in the **test split but never in train**. These are the items the model has never seen during training — at deploy time they would hit the `<NEW_BUSINESS>` OOV embedding. Phase 7 evaluates v2 (with `item_text_emb_pca32`, ColBERT-light) vs v1 (without) on this subset to measure how much the sentence-transformer item encoder rescues cold-start items.

In [11]:
train_biz_ids = set(train["business_id"].unique())
test_biz_ids = set(test["business_id"].unique())
cold_item_ids = test_biz_ids - train_biz_ids
print(f"train businesses: {len(train_biz_ids)}")
print(f"test businesses:  {len(test_biz_ids)}")
print(f"cold-start ITEMs (in test but never in train): {len(cold_item_ids)}")

coldstart_item_test = test[test["business_id"].isin(cold_item_ids)].copy()
coldstart_item_test.to_parquet(OUT_DIR / "coldstart_item_test_reviews.parquet", index=False)
print(f"cold-start ITEM reviews: {len(coldstart_item_test)}")

train businesses: 7540
test businesses:  8206
cold-start ITEMs (in test but never in train): 1471
cold-start ITEM reviews: 38387


Summary.

In [12]:
print(f"cities:           {CITIES}")
print(f"restaurants:      {len(restaurants)}")
print(f"reviews:          {len(reviews_rest)}")
print(f"users:            {len(users)}")
print(f"train/val/test:   {len(train)} / {len(val)} / {len(test)}")
print(f"cold-start USER:  {len(coldstart_test)}")
print(f"cross-city USER:  {len(crosscity_test)}")
print(f"cold-start ITEM:  {len(coldstart_item_test)} ({len(cold_item_ids)} unseen restaurants)")

cities:           ['Philadelphia', 'Tucson', 'Tampa']
restaurants:      9022
reviews:          1032056
users:            359007
train/val/test:   587676 / 83953 / 167908
cold-start USER:  140948
cross-city USER:  51720
cold-start ITEM:  38387 (1471 unseen restaurants)
